<a href="https://colab.research.google.com/github/faizcodesgo/clarityvoice/blob/main/voice-clone-chatterbox.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎙️ Clone Your Voice with EMOTION (Free) — Chatterbox

This replaces the old XTTS notebook. **Chatterbox** (Resemble AI, MIT license) clones your voice **and** gives real emotion + pacing control — the things XTTS couldn't do. It beat ElevenLabs in blind tests and runs on the free Colab GPU.

### How to use
1. **Runtime → Change runtime type → T4 GPU → Save**
2. Run each cell **once, top to bottom**.
3. Upload a **clean English** reference clip when asked (7-20s, natural tone, quiet room).
4. Two dials control everything:
   - **exaggeration** = emotion: `0.3` calm → `0.5` normal → `0.7+` dramatic
   - **cfg_weight** = pace: `~0.3` faster/energetic → `~0.5` normal → `~0.7` slower/deliberate

> If a cell shows an import error right after install: **Runtime → Restart session**, then run again from the **Load model** cell.

### 1. Check the GPU

In [1]:
!nvidia-smi -L

GPU 0: Tesla T4 (UUID: GPU-39e20be2-f021-2da2-3a77-a09f3fb50dee)


### 2. Install Chatterbox (~1-2 min)

In [ ]:
!pip install -q chatterbox-tts
print("Installed. If the NEXT cell shows an import error: Runtime > Restart session, then run from the Load model cell.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.5/108.5 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 53.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.2/24.2 MB 82.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 107.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 471.6/471.6 kB 38.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 766.6/766.6 MB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 94.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 107.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.3/58.3 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 107.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/2

### 3. Load the model (downloads weights first time, ~1 min)

In [ ]:
import torch, torchaudio as ta
from chatterbox.tts import ChatterboxTTS

device = "cuda" if torch.cuda.is_available() else "cpu"
model = ChatterboxTTS.from_pretrained(device=device)
print("Model loaded \u2705  (sample rate:", model.sr, ")")

### 4. Upload your English reference clip
One CLEAN clip, 7-20 seconds, natural conversational tone (not robotic reading), quiet room.

In [ ]:
from google.colab import files
print("Upload ONE clean English reference clip (7-20s).")
uploaded = files.upload()
REF = list(uploaded.keys())[0]
print("Reference:", REF)

### 5. Quick test — try the two dials
Change `text`, `exaggeration`, and `cfg_weight`, then run. Hear how the dials change the feel.

In [ ]:
from IPython.display import Audio, display

text = "Hello, this is my own voice. I think it finally sounds like me."
exaggeration = 0.6   # emotion: 0.3 calm -> 0.5 normal -> 0.7+ dramatic
cfg_weight   = 0.4   # pace:    0.3 faster -> 0.5 normal -> 0.7 slower

wav = model.generate(text, audio_prompt_path=REF, exaggeration=exaggeration, cfg_weight=cfg_weight)
ta.save("test.wav", wav, model.sr)
print("Done \u2705"); display(Audio("test.wav")); files.download("test.wav")

### 6. HUMAN pacing — different emotion + speed per line, with real pauses ✨
This is the magic: each line gets its own emotion + pace, and we insert **real silence** between lines.
Later, your **AI assistant's brain** will generate this list automatically. For now, edit the `script` below.

In [ ]:
import torch, torchaudio as ta
from IPython.display import Audio, display

# (text, exaggeration[emotion], cfg_weight[pace], pause_after_ms)
script = [
    ("So, here's where we are.",             0.5, 0.5, 350),
    ("Honestly, this is a great result!",     0.8, 0.35, 250),   # excited + faster
    ("But I want to be very clear.",          0.4, 0.6, 450),    # serious + slower + longer pause
    ("If we miss this deadline, it affects the whole release.", 0.55, 0.5, 0),
]

sr = model.sr
pieces = []
for txt, exag, cfg, pause_ms in script:
    w = model.generate(txt, audio_prompt_path=REF, exaggeration=exag, cfg_weight=cfg)
    pieces.append(w)
    if pause_ms > 0:
        pieces.append(torch.zeros(1, int(sr * pause_ms / 1000)))   # real silence = human pause

full = torch.cat(pieces, dim=1)
ta.save("my_voice_human.wav", full, sr)
print("Done \u2705 — emotion + speed vary per line, with real pauses between them.")
display(Audio("my_voice_human.wav")); files.download("my_voice_human.wav")

### 7. (Optional) Hindi — multilingual model
Meetings are English, so this is a bonus. Uses a separate multilingual model + a **Hindi** reference clip.
If this errors, just skip it (ping me) — your English path above is the main deliverable.

In [ ]:
# OPTIONAL Hindi. Upload a separate clean HINDI reference clip first.
from chatterbox.mtl_tts import ChatterboxMultilingualTTS
from google.colab import files
from IPython.display import Audio, display
import torchaudio as ta

print("Upload a clean HINDI reference clip:")
REF_HI = list(files.upload().keys())[0]

mtl = ChatterboxMultilingualTTS.from_pretrained(device=device)
wav = mtl.generate("\u0928\u092e\u0938\u094d\u0924\u0947, \u092f\u0939 \u092e\u0947\u0930\u0940 \u0905\u092a\u0928\u0940 \u0906\u0935\u093e\u091c\u093c \u0939\u0948\u0964",
                    language_id="hi", audio_prompt_path=REF_HI, exaggeration=0.5, cfg_weight=0.5)
ta.save("my_voice_hi.wav", wav, mtl.sr)
print("Done \u2705"); display(Audio("my_voice_hi.wav")); files.download("my_voice_hi.wav")